<a id="inizio"></a>
# 09 · Plotly per serie storiche

⏱ ~25 min &nbsp;·&nbsp; Dati: `../Dati/load_total_north_hourly_2024.xlsx`, `../Dati/TexasTurbine.csv`

**In questo notebook impariamo a**

- fare grafici a linee, istogrammi e scatter con Plotly Express
- confrontare più serie nello stesso grafico, partendo da dati wide-form
- riconoscere dati, mapping, tracce e layout di una figura

<a id="sez-1"></a>
## 1. Setup minimo

Restiamo sul minimo che serve nella pratica: `plotly.express`. Documentazione ufficiale:
[Plotly Express](https://plotly.com/python/plotly-express/),
[Line charts](https://plotly.com/python/line-charts/),
[Time series](https://plotly.com/python/time-series/).

In [ ]:
import pandas as pd
import plotly.express as px

<a id="sez-2"></a>
## 2. I quattro elementi chiave

Plotly diventa molto più leggibile se separi sempre quattro pezzi.

1. **Dati**: un DataFrame con una colonna tempo (`datetime`) ordinata.
2. **Mapping**: quali colonne finiscono su `x`, `y` (e, quando serve, `color`).
3. **Tracce**: una `Figure` è fatta di una o più tracce (`fig.data`). In una serie storica la traccia
   tipica è uno `Scatter` con `mode="lines"`.
4. **Layout e controlli**: titolo, assi, hover… si modificano con `fig.update_layout(...)`
   e `fig.update_xaxes(...)`.

<a id="sez-3"></a>
## 3. Il dataset stocks

`px.data.stocks()`, già incluso in Plotly, è un dataset
[wide-form](https://plotly.com/python/wide-form/): una colonna tempo (`date`) e una colonna per
ciascuna serie. I valori sono **indicizzati** (partono da 1): è comodo per confrontare andamenti relativi.

In [ ]:
df = px.data.stocks().copy()
df["date"] = pd.to_datetime(df["date"])
df.head()

> **Nota.**
>
> Se `date` non è `datetime` o non è ordinata, la linea può produrre risultati confusi (salti avanti
> e indietro).

<a id="sez-4"></a>
## 4. Primo grafico con Plotly Express

Il primo obiettivo è avere una linea leggibile con un mapping esplicito (`x` e `y`).

In [ ]:
azienda = "GOOG"

fig = px.line(
    df,
    x="date",
    y=azienda,
    title=f"{azienda} (indicizzato)"
)
fig.show()

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Cambia `azienda` con un'altra colonna del dataset (`AAPL`, `AMZN`, `MSFT`…). Poi limita il grafico
agli ultimi 12 mesi filtrando `df` prima di passarlo a `px.line`: il dataset finisce a fine 2019,
quindi bastano le date dal `2019-01-01`. Il grafico atteso parte da gennaio 2019.

</div>

In [ ]:
azienda = ...
mask_12_mesi = ...
fig_12_mesi = px.line(...)
fig_12_mesi.show()

La media mobile si calcola con `rolling` come nuova colonna e si disegna accanto alla serie
originale, passando le due colonne a `y`. Il dataset ha un valore a settimana.

In [ ]:
azienda = "GOOG"
finestra_mobile = 4

# media mobile come nuova colonna di un nuovo DataFrame
df_mobile = df[["date", azienda]].copy()
df_mobile[f"{azienda}_rolling"] = df_mobile[azienda].rolling(finestra_mobile).mean()
# attenzione ai valori mancanti all'inizio della serie: qui li togliamo
df_mobile = df_mobile.dropna().reset_index(drop=True)
fig = px.line(
    df_mobile,
    x="date",
    y=[azienda, f"{azienda}_rolling"],
    title=f"{azienda} con media mobile a {finestra_mobile} settimane"
)
fig.show()

<a id="sez-5"></a>
## 5. Confronto tra serie (wide-form)

Con i dati wide-form puoi passare direttamente una lista di colonne a `y`. È il caso che torna più
spesso: una tabella con molte serie già allineate nel tempo.

In [ ]:
aziende = ["AAPL", "AMZN", "MSFT"]

fig = px.line(
    df,
    x="date",
    y=aziende,
    title="Confronto tra serie (wide-form)"
)
fig.show()

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Aggiungi o rimuovi una società dalla lista `aziende`. Poi crea `df_indicizzato`, una copia di `df`
con le serie riportate a base 100 alla prima data (`valore / valore.iloc[0] * 100`), e ripeti il
grafico. Tutte le linee devono partire da 100.

</div>

In [ ]:
aziende = ["AAPL", "AMZN", "MSFT", ...]
df_indicizzato = df.copy()
df_indicizzato[aziende] = ...
fig_100 = px.line(...)
fig_100.show()

<a id="sez-6"></a>
## 6. Istogramma e scatter

Plotly Express ha una funzione per ogni tipo di grafico, con gli stessi argomenti di `px.line`. Le
proviamo sui dati orari della turbina eolica texana, con due colonne rinominate per comodità.

In [ ]:
turbina = pd.read_csv("../Dati/TexasTurbine.csv")
# il timestamp non ha l'anno: lo aggiungiamo noi
turbina["timestamp"] = pd.to_datetime("2023 " + turbina["Time stamp"], format="%Y %b %d, %I:%M %p")
turbina = turbina.rename(columns={"System power generated | (kW)": "potenza_kw", "Wind speed | (m/s)": "vento_ms"})
turbina.head(3)

L'istogramma conta quante ore cadono in ogni intervallo di velocità del vento: basta la `x`, e
`nbins` è il numero di barre.

In [ ]:
fig = px.histogram(turbina, x="vento_ms", nbins=30, title="Distribuzione del vento")
fig.show()

Lo scatter disegna un punto per riga e mostra la relazione tra due colonne: vento sulla x, potenza
sulla y. Ne esce la curva di potenza della turbina; `opacity` fa vedere dove i punti si accumulano.

In [ ]:
fig = px.scatter(turbina, x="vento_ms", y="potenza_kw", opacity=0.3, title="Vento e potenza")
fig.show()

<a id="sez-7"></a>
## 7. Esercizi

<a id="es-9-1"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 9.1 · Un mese di carico**

Il file di Terna ha il carico del Nord del 2024, un valore ogni quarto d'ora. Vogliamo guardare
gennaio da vicino.

**Cosa fare.**

1. Da `carico` tieni il solo gennaio 2024 in `gennaio`.
2. Costruisci `fig_carico` con `px.line`: `Total Load [MW]` sulla `Date`, titolo `Carico Nord, gennaio 2024`.

Output atteso: una linea sola.

</div>

In [ ]:
carico = pd.read_excel("../Dati/load_total_north_hourly_2024.xlsx")
carico = carico.sort_values("Date")

mask_gennaio = ...
gennaio = carico[mask_gennaio]

fig_carico = px.line(...)
fig_carico.show()

In [ ]:
from corso import verifica
verifica("9.1")

<a id="es-9-2"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 9.2 · La distribuzione della potenza**

Quante ore la turbina texana produce poco, e quante lavora vicino ai 3000 kW?

**Cosa fare.**

Costruisci in `fig_potenza` l'istogramma della colonna `potenza_kw` di `turbina`, con 30 barre e
titolo `Distribuzione della potenza`. Output atteso: la barra più alta è quella vicino a 0 kW.

</div>

In [ ]:
fig_potenza = px.histogram(...)
fig_potenza.show()

In [ ]:
from corso import verifica
verifica("9.2")

---

**Fine del notebook 09.** Precedente: [08 · Pandas: le date](08_Pandas_date.ipynb) &nbsp;·&nbsp; Prossimo: [E3 · Esercitazione 3](Esercitazione_3.ipynb)

<sub>Blocco 3 · Analisi dei dati con pandas e visualizzazione · Giornata 2 · Aula Avanzata</sub>